# Loading and Validating HMIE Datasets from Cloud Object Storage

`datamaite` can validate and load HMIE datasets directly from cloud object
storage -- `s3://`, `gs://`, or `az://` roots -- with no manual download
step. Annotation (JSON) checks stream directly from object storage, and
video integrity checks stream too: the probe opens each remote video as a
seekable file object and decodes through PyAV, so only the byte ranges it
actually reads (the container header plus a handful of sampled frames) are
transferred, not the whole file.

See the [Loading datasets from cloud object storage](../getting-started/cloud-storage.md)
guide for the full picture (credentials, `storage_options`, the CLI). This
notebook complements that guide with a runnable, end-to-end walkthrough.

**Why `memory://` instead of `s3://`?** This notebook uses fsspec's
built-in in-memory object store (`memory://`) so it executes anywhere with
zero credentials and zero network access -- including in CI and the docs
build. `memory://` is not a special case in `datamaite`: it is just another
non-local `fsspec` filesystem, so it exercises the *identical* remote code
path that `s3://`, `gs://`, and `az://` use. Everything demonstrated here
(streaming validation, streaming video probes, cloud-aware `load_mot`)
applies unchanged to a real bucket -- see the real-S3 section near the end.


In [ ]:
import os
from pathlib import Path

import datamaite

# major.minor.patch only, so the docs don't show a hatch-vcs dev suffix in CI
print(f"datamaite {'.'.join(datamaite.__version__.split('.')[:3])}")

# Resolve the shared example dataset the same way the other tutorials do.
# CI sets DATAMAITE_DATASETS_ROOT to the cloned `datasets/` directory;
# locally it falls back to a clone beside this notebook.
DATASETS_ROOT = Path(os.environ.get("DATAMAITE_DATASETS_ROOT", "datamaite-example-datasets/datasets"))
LOCAL_ROOT = DATASETS_ROOT / "hmie" / "valid"

local_files = sorted(p for p in LOCAL_ROOT.rglob("*") if p.is_file())
print(f"{len(local_files)} files under {LOCAL_ROOT}")

In [ ]:
import fsspec
from upath import UPath

# fsspec's in-memory filesystem is a process-global singleton -- start from a
# clean slate so re-running this notebook is deterministic.
fs = fsspec.filesystem("memory")
fs.store.clear()
fs.pseudo_dirs[:] = [""]

CLOUD_ROOT = UPath("memory://hmie-cloud-tutorial/valid")

# Upload every file, preserving the same relative layout datamaite expects
# on disk. A real cloud backend would use `UPath("s3://bucket/prefix", **storage_options)`
# here instead -- the mkdir/write_bytes calls below are identical either way.
for local_path in local_files:
    remote_path = CLOUD_ROOT / local_path.relative_to(LOCAL_ROOT)
    remote_path.parent.mkdir(parents=True, exist_ok=True)
    remote_path.write_bytes(local_path.read_bytes())

# The resulting object listing -- same layout as the local tree, just
# addressed by memory:// URLs instead of filesystem paths.
for obj in sorted(CLOUD_ROOT.rglob("*")):
    if obj.is_file():
        print(obj)

## Validate the dataset in place

`datamaite.validate` runs the full check suite -- folder structure,
annotation coverage, Scale spec compliance, and (with
`check_video_integrity=True`, the default) per-file video probes -- against
the `memory://` root exactly as it would against a local path or an
`s3://` URL.

**`workers=1` is a `memory://`-only constraint**, not a cloud-storage
limitation: fsspec's in-memory filesystem is a process-global singleton, so
a subprocess worker would see an empty store and find nothing to validate.
Real cloud backends (S3, GCS, Azure) have no such restriction -- point
`validate` at an `s3://` root and it parallelizes across `workers` worker
processes exactly like a local dataset.


In [ ]:
result = datamaite.validate(str(CLOUD_ROOT), workers=1, check_video_integrity=True)

print(result.summary(use_color=False))

## Load it as a MOT dataset

`datamaite.load_mot` reads the same `memory://` root into an in-memory
`BoxTrackDataset`. Passing `require_video=True` forces a real video probe
per sequence -- on `memory://` this decodes through PyAV against the
in-memory file object; on `s3://`/`gs://`/`az://` the identical code path
streams ranged reads from the bucket. Either way, the probe is what flips
`num_frames_exact` to `True`: without it, `num_frames` is only an
annotation-derived lower-bound estimate.


In [ ]:
dataset = datamaite.load_mot(str(CLOUD_ROOT), dataset_format="hmie", require_video=True)

print(f"{dataset.sequence_count} sequence(s)\n")

header = f"{'annotation':45s} {'frames':>7s} {'fps':>6s} {'size':>11s} {'exact':>6s}"
print(header)
print("-" * len(header))
for seq in dataset.sequences:
    name = Path(seq.annotation_path).name
    size = f"{seq.width}x{seq.height}" if seq.width and seq.height else "-"
    print(f"{name:45s} {seq.num_frames!s:>7s} {seq.fps:>6.1f} {size:>11s} {seq.num_frames_exact!s:>6s}")

all_exact = all(seq.num_frames_exact for seq in dataset.sequences)
print(f"\nall frame counts exact (from streamed probes): {all_exact}")
if not all_exact:
    raise RuntimeError("expected require_video=True to yield exact frame counts")

## The real-S3 equivalent

Everything above runs unchanged against a real bucket -- swap the
`memory://` root for an `s3://` (or `gs://`/`az://`) URL, install the
matching backend extra, and pass credentials via `storage_options` or the
provider's usual environment variables:

```python
# pip install "datamaite[aws]"   # or [gcs] / [azure] / [cloud] for all three
import datamaite

result = datamaite.validate(
    "s3://my-bucket/datasets/hmie/valid",
    workers=8,  # full parallelism -- no memory://-style singleton constraint
    storage_options={
        "key": "...",
        "secret": "...",
        "client_kwargs": {"endpoint_url": "https://..."},
    },
)
print(result.summary())

dataset = datamaite.load_mot(
    "s3://my-bucket/datasets/hmie/valid",
    dataset_format="hmie",
    require_video=True,
    storage_options={"key": "...", "secret": "..."},
)
```

`storage_options` is optional -- if omitted, credentials resolve the same
way as any `fsspec` application (`AWS_ACCESS_KEY_ID` / `AWS_SECRET_ACCESS_KEY`,
`GOOGLE_APPLICATION_CREDENTIALS`, `AZURE_STORAGE_ACCOUNT_KEY`, etc.). The
CLI accepts the same URLs and has no credentials flag -- configure the
environment instead:

```bash
datamaite validate s3://my-bucket/datasets/hmie/valid --no-cache
```

This block is intentionally not executed: it targets a real bucket and
would either need live credentials or fail the hermetic docs build.


## What travels over the network

Only the bytes each check actually needs: annotation JSON is read directly,
and video probes transfer just the container header plus a handful of
sampled frames -- never a full-file download, and no temporary files or
presigned URLs. Validation findings always report the dataset's logical
path (the `s3://...` URL), regardless of backend.

For cache behavior at scale (fingerprinting cost, `--skip-video-check` for
JSON-only runs), see the
[Notes at scale](../getting-started/cloud-storage.md#notes-at-scale)
section of the getting-started guide.


<hr>